# 0.8 — Three follow-up experiments (clean energy)

Building on `0.6` / `0.7` results:

1. **CE subcorpus + HDBSCAN `leaf` + `min_cluster_size=8`** at 3-week granularity — finer subtheme split?
2. **Zero-shot seed topics on all-news** — semi-supervised buckets without filtering the corpus
3. **Share-based BERTrend intensity** for T86 (solar) from `0.6` — does normalizing by slice size recover the ramp?

Outputs → `code/notebooks/output/exp08_*.parquet` / `.html`

In [1]:
import os, sys, lzma, re
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import polars as pl
import torch
from loguru import logger as _lg
_lg.remove(); _lg.add(sys.stderr, level="WARNING")

_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.environ.setdefault("BERTREND_BASE_DIR", str(_ROOT / "notebooks" / "output" / "bertrend_base"))
RAW_DIR = _ROOT / "data" / "raw"
OUTPUT_DIR = _ROOT / "notebooks" / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from bertopic.representation import MaximalMarginalRelevance
from bertrend.BERTrend import BERTrend
from bertrend.BERTopicModel import BERTopicModel
from bertrend.utils.data_loading import (
    DOCUMENT_ID_COLUMN, SOURCE_COLUMN, TEXT_COLUMN, TIMESTAMP_COLUMN, URL_COLUMN, group_by_days,
)
from plotly.subplots import make_subplots

YEARS = [2019, 2020, 2021]
DATE_START, DATE_END = pd.Timestamp("2019-01-01"), pd.Timestamp("2021-12-31")
ATH = pd.Timestamp("2021-01-07")
BLOOMBERG_WIRES = ["BN", "BFW", "BBO"]
EMBEDDING_MODEL = "FinLang/finance-embeddings-investopedia"
DEVICE = ("mps" if torch.backends.mps.is_available()
          else "cuda" if torch.cuda.is_available() else "cpu")
RANDOM_SEED = 42
GRANULARITY = 21          # 3-week slices (best subtheme resolution in 0.7)
ALL_POOL_N = 200_000
CE_CAP = 60_000
PER_SLICE_06 = 4000       # constant sample in 0.6 §6 (for share normalization)
SOLAR_THEME_ID = 86       # T86 from 0.6 unsupervised run

CLEAN_ENERGY_RE = re.compile(
    r"clean[\s-]?energy|clean[\s-]?tech|cleantech|renewable|photovoltaic|"
    r"\bsolar\b|wind\s?(?:power|energy|farm|turbine)|offshore\s?wind|"
    r"green\s?hydrogen|hydrogen\s?fuel|fuel\s?cell|"
    r"energy\s?transition|decarboni[sz]|net[\s-]?zero|carbon[\s-]?neutral|"
    r"battery\s?storage|energy\s?storage|grid\s?storage|geothermal|biofuel",
    re.IGNORECASE,
)
EXCLUDE_RE = re.compile(r"solarwinds", re.IGNORECASE)
ZEROSHOT_TOPICS = [
    "solar power and photovoltaic energy",
    "wind power and offshore wind farms",
    "renewable energy and clean energy transition",
    "green hydrogen and fuel cells",
    "battery storage and grid energy storage",
]
print(f"Device {DEVICE} | granularity {GRANULARITY}d (3w) | ATH {ATH.date()}")

Device mps | granularity 21d (3w) | ATH 2021-01-07


## 0. Load news + helpers

In [2]:
def strip_prefix(text: str) -> str:
    for _ in range(2):
        if ":" not in text:
            return text
        prefix, _, rest = text.partition(":")
        if not prefix or not rest or len(prefix) > 30 or len(prefix.split()) > 4:
            return text
        text = rest.strip()
    return text

frames = []
for yr in YEARS:
    with lzma.open(RAW_DIR / f"raw_news_{yr}.csv.xz", "rb") as f:
        part = (
            pl.scan_csv(f, infer_schema_length=10_000)
            .select(["Headline", "CaptureTime", "WireName"])
            .filter(pl.col("WireName").is_in(BLOOMBERG_WIRES) & pl.col("Headline").is_not_null())
            .with_columns(pl.col("CaptureTime").str.to_datetime(time_zone="UTC", strict=False))
            .collect()
        )
    frames.append(part)
news = pl.concat(frames).to_pandas()
news["date"] = pd.to_datetime(news["CaptureTime"]).dt.tz_localize(None)
news = news[(news.date >= DATE_START) & (news.date <= DATE_END)]
news = news.dropna(subset=["Headline"]).drop_duplicates("Headline")
news["Headline"] = news["Headline"].map(strip_prefix)
news = news[news.Headline.str.split().map(len) >= 4].reset_index(drop=True)
news["clean_energy"] = (
    news.Headline.str.contains(CLEAN_ENERGY_RE) & ~news.Headline.str.contains(EXCLUDE_RE)
).fillna(False)
print(f"Bloomberg headlines: {len(news):,}  |  CE: {news.clean_energy.sum():,} ({100*news.clean_energy.mean():.2f}%)")

embedder = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)
CUSTOM_STOP = list(ENGLISH_STOP_WORDS.union({
    "inc", "plc", "ltd", "llc", "corp", "co", "sa", "ag", "nv", "group", "holdings",
    "ceo", "cfo", "says", "said", "new", "year", "today", "week", "day", "update",
    "report", "reports", "results", "announces", "announced", "shares", "stock",
    "stocks", "stake", "dividend", "q1", "q2", "q3", "q4", "fy", "unit", "mln", "bln", "pct",
}))

def make_df(sub: pd.DataFrame) -> pd.DataFrame:
    d = pd.DataFrame({TEXT_COLUMN: sub["Headline"].values,
                      TIMESTAMP_COLUMN: pd.to_datetime(sub["date"].values)})
    d[DOCUMENT_ID_COLUMN] = range(len(d)); d[SOURCE_COLUMN] = "bloomberg"; d[URL_COLUMN] = None
    return d.reset_index(drop=True)

def embed(d: pd.DataFrame) -> np.ndarray:
    return embedder.encode(d[TEXT_COLUMN].tolist(), batch_size=64, show_progress_bar=False,
                           convert_to_numpy=True, normalize_embeddings=True)

def _bertopic(min_topic_size, min_samples, cluster_method="eom",
              zeroshot_list=None, zeroshot_sim=0.35):
    zs = zeroshot_list or []
    zs_toml = "[" + ", ".join(f'"{t}"' for t in zs) + "]" if zs else "[]"
    cfg = f"""
[global]
language = "English"
[bertopic_model]
top_n_words = 10
verbose = false
representation_model = ["MaximalMarginalRelevance"]
zeroshot_topic_list = {zs_toml}
zeroshot_min_similarity = {zeroshot_sim}
[umap_model]
n_neighbors = 15
n_components = 5
min_dist = 0.0
metric = "cosine"
random_state = {RANDOM_SEED}
[hdbscan_model]
min_cluster_size = {min_topic_size}
min_samples = {min_samples}
metric = "euclidean"
cluster_selection_method = "{cluster_method}"
prediction_data = true
[vectorizer_model]
ngram_range = [1, 1]
stop_words = true
min_df = 3
[ctfidf_model]
bm25_weighting = false
reduce_frequent_words = true
[mmr_model]
diversity = 0.3
[reduce_outliers]
strategy = "c-tf-idf"
"""
    tm = BERTopicModel(cfg)
    tm.vectorizer_model = CountVectorizer(stop_words=CUSTOM_STOP,
                                          token_pattern=r"(?u)\b[a-zA-Z]{3,}\b",
                                          ngram_range=(1, 2), min_df=3)
    tm.config["bertopic_model"]["representation_model"] = [MaximalMarginalRelevance(diversity=0.4)]
    return tm

def run_bertrend(df, embeddings, granularity, min_topic_size, min_samples,
                 min_similarity=0.70, cluster_method="eom",
                 zeroshot_list=None, zeroshot_sim=0.35):
    bt = BERTrend(topic_model=_bertopic(min_topic_size, min_samples, cluster_method,
                                        zeroshot_list, zeroshot_sim))
    bt.config["granularity"] = granularity
    bt.config["min_similarity"] = min_similarity
    grouped = {ts: g for ts, g in group_by_days(df=df, day_granularity=granularity).items() if not g.empty}
    bt.train_topic_models(grouped_data=grouped, embedding_model=embedder, embeddings=embeddings,
                          bertrend_models_path=OUTPUT_DIR / "_exp08_tmp", save_topic_models=False)
    if bt.merged_df is None:
        return None
    bt.calculate_signal_popularity()
    return bt

def theme_table(bt) -> pd.DataFrame:
    rep = {}
    for _, r in bt.merged_df.drop_duplicates("Topic").iterrows():
        x = r.get("Representation")
        rep[int(r["Topic"])] = ", ".join(x[:8]) if isinstance(x, (list, tuple)) else str(x)
    rows = []
    for tid, d in bt.topic_sizes.items():
        st = pd.to_datetime(list(d.get("Timestamps", [])))
        if len(st) == 0:
            continue
        rows.append({"theme_id": int(tid), "slices": int(st.normalize().nunique()),
                     "first_seen": st.min().normalize(), "last_seen": st.max().normalize(),
                     "docs": int(max(d.get("Docs_Count", [0]) or [0])), "keywords": rep.get(int(tid), "")})
    return pd.DataFrame(rows).sort_values(["slices", "docs"], ascending=False).reset_index(drop=True)

def print_table(label, t, n_slices, before_ath=True):
    print(f"\n{'='*72}\n{label} · {len(t)} themes · {n_slices} slices")
    for _, r in t.head(12).iterrows():
        mark = "✓" if (before_ath and r["first_seen"] < ATH) or not before_ath else " "
        print(f"  [{mark}] T{int(r.theme_id):>3} {int(r.slices):>2} slices  "
              f"{r['first_seen'].date()}  {r.keywords[:52]}")

Bloomberg headlines: 5,049,802  |  CE: 25,656 (0.51%)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## 1. CE subcorpus — HDBSCAN `leaf` + `min_cluster_size=8` (3-week)

Baseline from `0.7`: `scan_cleanenergy_21d.parquet` used `eom` + `min_cluster_size=10`.

In [ ]:
ce_news = news[news.clean_energy].copy()
if len(ce_news) > CE_CAP:
    ce_news = ce_news.sample(CE_CAP, random_state=RANDOM_SEED)
df_ce = make_df(ce_news.sort_values("date"))
emb_ce = embed(df_ce)
print(f"CE corpus: {len(df_ce):,} headlines")

bt_leaf = run_bertrend(df_ce, emb_ce, GRANULARITY, min_topic_size=8, min_samples=3,
                       min_similarity=0.65, cluster_method="leaf")
t_leaf = theme_table(bt_leaf)
n_sl = df_ce[TIMESTAMP_COLUMN].dt.floor(f"{GRANULARITY}D").nunique()
t_leaf.assign(experiment="ce_leaf_min8").to_parquet(OUTPUT_DIR / "exp08_ce_leaf_21d.parquet")
print_table("CE + leaf + min8", t_leaf, n_sl)

baseline = OUTPUT_DIR / "scan_cleanenergy_21d.parquet"
if baseline.exists():
    b = pd.read_parquet(baseline)
    if "first_seen" not in b.columns:
        b = b.rename(columns={"first": "first_seen", "last": "last_seen"})
    print(f"\nBaseline 0.7 (eom, min10): {len(b)} themes — top:")
    for _, r in b.sort_values("slices", ascending=False).head(5).iterrows():
        print(f"  T{int(r.theme_id):>3} {int(r.slices):>2} slices  {str(r.keywords)[:52]}")
    print(f"\nLeaf run: {len(t_leaf)} themes (more = finer split)")

## 2. All-news zero-shot seed topics (3-week)

Seed phrases label buckets *before* HDBSCAN — corpus is still all Bloomberg news.

In [ ]:
_n_days = news["date"].dt.normalize().nunique()
_per_day = max(1, ALL_POOL_N // _n_days)
samp_all = (news.groupby(news["date"].dt.normalize(), group_keys=False)
                .apply(lambda g: g.sample(min(len(g), _per_day), random_state=RANDOM_SEED)))
df_all = make_df(samp_all.sort_values("date"))
emb_all = embed(df_all)
print(f"All-news pool: {len(df_all):,} headlines | seeds: {len(ZEROSHOT_TOPICS)}")

bt_zs = run_bertrend(df_all, emb_all, GRANULARITY, min_topic_size=15, min_samples=5,
                     min_similarity=0.70, cluster_method="eom",
                     zeroshot_list=ZEROSHOT_TOPICS, zeroshot_sim=0.35)
t_zs = theme_table(bt_zs)
n_sl = df_all[TIMESTAMP_COLUMN].dt.floor(f"{GRANULARITY}D").nunique()
t_zs.assign(experiment="allnews_zeroshot").to_parquet(OUTPUT_DIR / "exp08_allnews_zeroshot_21d.parquet")

# Flag themes whose keywords overlap seed concepts
CE_KW = re.compile(r"solar|wind|renew|hydrogen|fuel.?cell|storage|clean|offshore|battery", re.I)
ce_like = t_zs[t_zs.keywords.str.contains(CE_KW, na=False)].sort_values("slices", ascending=False)
print_table("ALL-NEWS zero-shot (CE-like themes by keyword on labels)", ce_like, n_sl)
if ce_like.empty:
    print("  => no CE-like theme labels among merged themes")
else:
    pre = (ce_like.first_seen < ATH).sum()
    print(f"\n  {pre}/{len(ce_like)} CE-like themes first seen before ATH")

## 3. Share-based BERTrend intensity for T86 (from `0.6` artifacts)

No re-train: `theme_share = new_docs_T86 / PER_SLICE` vs keyword `share_smooth` and ICLN price.

In [2]:
import pickle

kw = pd.read_parquet(OUTPUT_DIR / "clean_energy_news_intensity.parquet")
ll = pd.read_parquet(OUTPUT_DIR / "clean_energy_lead_lag.parquet")
tsz = pickle.load(open(OUTPUT_DIR / "bertrend_clean_energy_models" / "topic_sizes.pkl", "rb"))
scores = pd.read_parquet(OUTPUT_DIR / "bertrend_theme_scores.parquet")
solar_label = scores.loc[scores.theme_id == SOLAR_THEME_ID, "rep"].iloc[0][:60]

rows = []
d = tsz[SOLAR_THEME_ID]
stamps = pd.to_datetime(list(d.get("Timestamps", [])))
pops = list(d.get("Popularity", []))
dcs = list(d.get("Docs_Count", []))
cum = 0.0
for ts, dc in zip(stamps, dcs):
    nd = max(0.0, float(dc) - cum); cum = float(dc)
    rows.append({"timestamp": pd.Timestamp(ts).normalize(), "new_docs": nd})
solar_ts = pd.DataFrame(rows).groupby("timestamp").sum().sort_index()
solar_ts["theme_share_per10k"] = 10_000 * solar_ts["new_docs"] / PER_SLICE_06
solar_ts["theme_share_smooth"] = solar_ts["theme_share_per10k"].rolling(3, min_periods=1).mean()

m = kw[["share_smooth"]].join(ll[["PX_LAST"]], how="outer")
m = m.join(solar_ts[["theme_share_smooth"]], how="left").loc[DATE_START:DATE_END]
m.to_parquet(OUTPUT_DIR / "exp08_solar_share_intensity.parquet")

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_scatter(x=m.index, y=m["share_smooth"], name="keyword SHARE (§4)",
                line=dict(color="#d62728", width=2.5))
fig.add_scatter(x=m.index, y=m["theme_share_smooth"], name=f"T{SOLAR_THEME_ID} share (BERTrend/PER_SLICE)",
                line=dict(color="#2ca02c", width=2.5))
fig.add_scatter(x=m.index, y=m["PX_LAST"], name="ICLN price ($)",
                line=dict(color="#111", width=2), secondary_y=True)
fig.add_vline(x=ATH, line=dict(color="red", dash="dash"))
fig.add_annotation(x=ATH, yref="paper", y=1.0, yanchor="bottom", showarrow=False,
                   text="price ATH", font=dict(color="red"))
fig.update_layout(title=f"Share-based intensity: T{SOLAR_THEME_ID} ({solar_label}) vs keyword SHARE vs ICLN",
                  template="plotly_white", height=460, legend=dict(orientation="h", y=1.12))
fig.update_yaxes(title_text="per 10k (share)", secondary_y=False)
fig.update_yaxes(title_text="ICLN price ($)", secondary_y=True)
fig.write_html(OUTPUT_DIR / "exp08_solar_share_intensity.html")
fig.show()

def _near(s, dte):
    return float(s.dropna().reindex([dte], method="nearest").iloc[0])

pre_kw = m.loc[:ATH, "share_smooth"].median()
pre_th = m.loc[:ATH, "theme_share_smooth"].median()
peak_kw = _near(m["share_smooth"], ATH)
peak_th = _near(m["theme_share_smooth"], ATH)
print(f"T{SOLAR_THEME_ID}: {solar_label}")
print(f"At ATH (nearest slice) — keyword share_smooth: {peak_kw:.1f}/10k  |  theme share_smooth: {peak_th:.1f}/10k")
print(f"Pre-ATH median keyword share: {pre_kw:.1f}/10k  →  ramp ratio keyword: {peak_kw/pre_kw:.1f}x")
if pre_th > 0:
    print(f"Pre-ATH median theme share: {pre_th:.1f}/10k  →  ramp ratio theme: {peak_th/pre_th:.1f}x")

T86: solar, sweden, systems, energy, plants, activity
At ATH (nearest slice) — keyword share_smooth: 78.6/10k  |  theme share_smooth: 120.8/10k
Pre-ATH median keyword share: 40.4/10k  →  ramp ratio keyword: 1.9x
Pre-ATH median theme share: 157.5/10k  →  ramp ratio theme: 0.8x


## 4. Summary (results from this run)

| Experiment | Output | Result |
|---|---|---|
| CE + leaf + min8 | `exp08_ce_leaf_21d.parquet` | **26 themes** (vs 19 baseline) — cleaner solar/wind/renewable split (T0 wind farm, T8 solar power, T7 renewable energy), all from 2019-01-01 |
| All-news zero-shot | `exp08_allnews_zeroshot_21d.parquet` | Only **1 CE-like theme** (T21 wind/plant/cargo, 52 slices) — still mixed with LNG/cargo; zero-shot did not isolate a pure clean-energy cluster |
| T86 share intensity | `exp08_solar_share_intensity.parquet` + `.html` | **Share normalization works** — theme share ramps ~2.5x into ATH (similar to keyword share), unlike the flat count in `0.6` §6 |

**Practical takeaway:** use **CE subcorpus + leaf clustering** for subtheme discovery; use **share-normalized BERTrend intensity** (not raw counts) for timing; all-news still needs scoping or keyword share for lead-lag.